# CV Masterclass 06: Self-Supervised & Contrastive Vision

Welcome to Notebook 06. So far, every model we have discussed required supervised learning: an image of a dog explicitly paired with a human-annotated label `Y="Dog"`.

But human annotation doesn't scale. Meta and Google have trillions of unlabelled images. How do you train a CNN on an image with no label?

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"In SimCLR (Contrastive Learning), if you train on a small batch size of 32 images, the network fails completely to learn any visual intelligence. But if you increase the batch size to 4,096 images, it approaches Supervised accuracy. Why does the math of the InfoNCE denominator demand massive batch sizes?"*

---

## Table of Contents
1. **The Representation Problem:** Pre-training without labels.
2. **Siamese Networks:** Learning by pushing and pulling.
3. **SimCLR & InfoNCE Math:** The necessity of Negative Samples.
4. **DINO Integration:** How Self-Supervised ViTs learn segmentation natively.


## 1. The Representation Problem

Imagine an algorithm that takes an image $x$, passes it through a CNN, and outputs a 128-dimensional vector $v$. We want $v$ to capture the core "essence" or "representation" of the image.

Without labels, how do we force the CNN to output a good vector?

**The Augmentation Trick:** 
We take a single unlabelled image of a car. We apply two random, brutal image augmentations to it (e.g., Crop + Heavy Color Jitter for Image A, and Rotate + Gaussian Blur for Image B).

To a pixel-matching algorithm, Image A and Image B look completely different. But they both originated from the exact same car.

## 2. Siamese Networks

We pass Image A through our CNN to get vector $v_A$. We pass Image B through the *exact same* CNN to get vector $v_B$.

Because they came from the same source image, we mathematically instruct the network to **maximize the cosine similarity** between $v_A$ and $v_B$. We "pull" them together in the 128-dimensional hyperspace.

**The Catastrophic Collapse:**
If we only pull things together, the CNN will realize: *"Oh, this is easy. I will just output the exact same vector `[0, 0, 0... 0]` for every single image in the universe."*
If every image outputs `0`, the distance between $v_A$ and $v_B$ is perfectly zero! The loss is zero. The network learned absolutely nothing.

We must introduce **Negative Samples** to prevent collapse.

In [ ]:
import torch
import torch.nn.functional as F

# -----------------------------------------------------
# IMPLEMENTATION: The InfoNCE Loss (SimCLR)
# -----------------------------------------------------

def info_nce_loss(features, temperature=0.1):
    """
    A simplified InfoNCE loss function. 
    Assume 'features' is a tensor of shape [2*N, D] where N is batch size.
    The first N features are Augmentation A. The second N are Augmentation B.
    Therefore, feature i and feature i+N are positive pairs.
    """
    # Normalize features to the unit hypersphere
    features = F.normalize(features, dim=1)
    
    # Calculate Cosine Similarity between ALL pairs in the entire batch
    # Shape: [2N, 2N]
    similarity_matrix = torch.matmul(features, features.T)
    
    # Now apply the mathematical formula for InfoNCE
    # We want to maximize the diagonal + offset (Positives)
    # While pushing away everything else in the matrix (Negatives)
    
    # (Implementation abstracted for mathematical clarity in curriculum)
    print("Calculating InfoNCE: Numerator pulls Positive Pairs... Denominator repels ALL Negative Pairs in the batch.")

info_nce_loss(torch.randn(64, 128))

## 3. SimCLR & InfoNCE Math

To prevent CNN collapse, the **InfoNCE Loss** function does two things simultaneously:
1. **Numerator:** Pulls the Positive Pair ($v_A$ and $v_B$) together.
2. **Denominator:** Pushes $v_A$ explicitly *away* from every single other vector in the current Batch.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does SimCLR demand massive batch sizes (like 4,096) to work well?*

**A:** 
Negative Samples only come from the other images in the *current batch*. 

If your batch size is 16, $v_A$ is only repelled by 15 random images. Pushing a vector away from 15 random vectors does not create a structured, highly uniform 128-dimensional hyperspace. 

If your batch size is 4,096, $v_A$ is violently repelled by 4,095 diverse images. The denominator of InfoNCE iterates over all 4,095 negatives, forcing the network to carve out incredibly specific, minute, highly intelligent feature representations just to keep $v_A$ safely away from everything else.

## 4. DINO Integration (Self-Supervised ViTs)

When you combine Self-Supervised Learning (like DINO by Meta) with a Vision Transformer (Notebook 05), something spectacular happens.

Because the ViT cuts the image into patches, and the Self-Supervised loss forces the network to find the "essence" of the image regardless of crops, the ViT's internal Attention Heads mathematically learn to perfectly outline the main object.

A DINO ViT trained entirely on unlabelled data natively outputs nearly flawless **Instance Segmentation masks** simply by looking at the pure attention maps of the `[CLS]` token. The network discovered the concept of object boundaries entirely on its own.